In [ ]:
import pandas as pd


In [ ]:
#load the dataset
df  = pd.read_csv("C:\\Users\\user\\Downloads\\Telco-Customer-Churn.csv")
print("shape:" ,df.shape)
print("first five: ",df.head(2))
df.info()

In [ ]:
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True) * 100)

In [ ]:
print("missing Values:")
print(df.isnull().sum())

print("\nTotalCharges data type:")
print(df["TotalCharges"].dtype)

print("\nBlank TotalCharges values:")
print((df["TotalCharges"].str.strip() == "").sum())

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Missing TotalCharges after conversion:")
print(df["TotalCharges"].isnull().sum())

print(df[df["TotalCharges"].isnull()][["tenure", "MonthlyCharges", "TotalCharges"]])

In [ ]:
#remove records with missing Total charges
df = df.dropna(subset=["TotalCharges"])

print("Dataset shape after cleaning:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
print(df.columns.tolist())

In [ ]:
#the input feature
X = df.drop(columns = ["Churn", "customerID"])
#the target to we predict
y = df["Churn"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget examples:")
print(y.head())

In [ ]:
#split the data before the preprocessing of encording the categorical 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
#identifying categorical feature

categorical_features = X_train.select_dtypes(
    include=["str", "object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include = ['number']
).columns.tolist()

print("Categorical features:", categorical_features)
print("Number of Categorical features:", len(categorical_features))

print("numerical features:", numerical_features)
print("Number of numerical features:", len(numerical_features))

In [ ]:
#encoding libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers =[(
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    )],
    remainder = "passthrough"
)


In [ ]:
#training the model from training data

X_train_encoded = preprocessor.fit_transform(X_train)

#transform test data using the same fitted method
X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:" , X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

In [ ]:
print("Original categorical data:")
print(X_train[categorical_features].head(2))

In [ ]:
#to see the encoded one
encoded_columns = preprocessor.get_feature_names_out()
print("Encoded columns names:")
print(encoded_columns)

print("\nFirst 5 endoded records:")
print(X_train_encoded[:5])



In [ ]:
#designing a decision tree model to predict the churn 
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train_encoded, y_train)
print("model training completed")
# the fit methods learns the pattern from the encoded training feature and there corresponding chun lable

In [ ]:
#Model prediction
y_pred= model.predict(X_test_encoded)

print("First 10 actual outcomes:")
print("y_test data:",y_test.head(10).tolist())

print("\nFirst 10 predicted outcomes:")
print("The predicted values:", y_pred[:10])

In [ ]:
#we evaluate with the model using the confussion metrics
from sklearn.metrics import(accuracy_score,confusion_matrix,classification_report)

print("Accuracy:",accuracy_score(y_test,y_pred))

print("\nConfussion Matrix:")
print(confusion_matrix(y_test,y_pred))

print("\nClassfication Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer,recall_score


#define lables

churn_recall = make_scorer(
    recall_score,
    pos_label ="Yes"
)


depths = [3,5,7,10,15]

for depth in depths:
    model_cv = DecisionTreeClassifier(
        max_depth = depth,
        random_state= 42

    )

    scores = cross_val_score(
        model_cv,
        X_train_encoded,
        y_train,
        cv=5,
        scoring = churn_recall,
        n_jobs = -1
        )
    print(
        f"Depth: {depth:2d} | " 
        f"mean recall: :{scores.mean():.3f} | "
        f"std: {scores.std():.3f}"
          
          
          )

In [ ]:
#examine the precision so as to know realy how many incorectly flags were made
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

best_model = DecisionTreeClassifier(
    max_depth= 7,
    random_state=42
)
best_model.fit(X_train_encoded, y_train)

y_pred_best = best_model.predict(X_test_encoded)

print("confusion matrix:")
print(confusion_matrix(y_test,y_pred_best))

print("\nClassfication Report:")
print(classification_report(y_test,y_pred_best))

In [ ]:
#lets use the algorithmn for random forest for the good generalization
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state= 42,
    class_weight = "balanced",
    n_jobs =-1

)
rf_model.fit(X_train_encoded,y_train)
y_pred_rf = rf_model.predict(X_test_encoded)

print("confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification report:")
print(classification_report(y_test,y_pred_rf))

In [ ]:
#on businedd rules lets create a pipeline so that every algorithm learn by avoiding inf from validatio
# fold from influencing preprocessing,, aldo data leakage

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
#preprocess categorical features

preprocessor = ColumnTransformer(
    transformers=[(
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    )],
    remainder="passthrough"
)

#Decision tree
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",DecisionTreeClassifier(
        max_depth= 7,
        random_state= 42
    ))

])

#Decision tree
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",RandomForestClassifier(
        n_estimators = 200,
        class_weight = "balanced",
        n_jobs = 1,
        random_state= 42
    ))

])
print("Both Pipelines created successfully.")






In [ ]:
#Validate the score for both pipelines
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score,f1_score

scoring ={
    "accuracy":"accuracy",
    "precision": make_scorer(precision_score, pos_label="Yes"),
    "recall":make_scorer(recall_score,pos_label= "Yes"),
    "f1":make_scorer(f1_score,pos_label="yes")
}
models= {
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline
}

for name,pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring = scoring,
        n_jobs=-1
    )
    print("\n{name}")
    print("-" *35)

    for metric in scoring:
        mean_score = scores[f"test_{metric}"].mean()
        std_score = scores[f"test_{metric}"].std()
        print(
            f"{metric.capitalize():<12}:"
            f"{mean_score:.3f} (+/- {std_score:.3f})"
        )
